# PyTorch - Data Preparation

The notebook experiments with the two main PyTorch primitives to work with data:
- `torch.utils.data.DataLoader` - It wraps an iterable around a `Dataset`
- `torch.utils.data.Dataset` - It stores samples and corresponding labels

PyTorch offers domain-specific libraries, such as:
- TorchText
- TorchVision
- TorchAudio

Each includes `Datasets` for specific domains.

# Notebook Setup

## Imports

In [1]:
# Import Standard Libraries
import os
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets
from torchvision.io import read_image
from torchvision.transforms import ToTensor

# Dataset

## Default Dataset

In [2]:
# Load FashionMNIST data
fashion_mnist_train = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)
fashion_mnist_test = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

print('FashionMNIST Train Object Type: ', type(fashion_mnist_train))
print('FashionMNIST Test Object Type: ', type(fashion_mnist_test))

FashionMNIST Train Object Type:  <class 'torchvision.datasets.mnist.FashionMNIST'>
FashionMNIST Test Object Type:  <class 'torchvision.datasets.mnist.FashionMNIST'>


## Custom Dataset

They have to implement three functions:
- `__init__`
- `__len__`
- `__getitem__`

In [ ]:
class CustomImageDataset(Dataset):
    def __init__(self, annotations_file, image_directory, image_transformation=None, label_transformation=None):
        self.image_labels = pd.read_csv(annotations_file)
        self.image_directory = image_directory
        self.image_transformation = image_transformation
        self.label_transformation = label_transformation

    def __len__(self):
        return len(self.image_labels)

    def __getitem__(self, index):
        # Retrieve image path
        image_path = os.path.join(self.image_directory, self.image_labels.iloc[index, 0])

        # Read image and corresponding label
        image = read_image(img_path)
        label = self.image_labels.iloc[index, 1]

        # Apply transformation to the image and label (if necessary)
        if self.image_transformation:
            image = self.image_transformation(image)
        if self.label_transformation:
            label = self.label_transformation(label)
        return image, label

## Custom Dataset with Transform Function

In [14]:
# Create sample data
features = pd.DataFrame({
    'feature_1': [2, 2, 2],
    'feature_2': [3, 3, 3],
})
labels = pd.DataFrame({
    'label': [1, 1, 1]
})

# Define function to engineer a 'feature_3' = 'feature_1' * 'feature_2'
def compute_feature(data: pd.Series) -> pd.Series:
    data['feature_3'] = data['feature_1'] * data['feature_2']
    return data

# Define a custom dataset with a transform function as 'compute_feature'
class CustomDatasetWithTransform(Dataset):
    def __init__(self, features: pd.DataFrame, labels: pd.DataFrame):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        feature_sample = self.features.iloc[index, :]
        label_sample = self.labels.iloc[index, :]
        return compute_feature(feature_sample), label_sample

# Instance the dataset and retrieve the first element to check the feature transformation
custom_dataset_with_transform_example = CustomDatasetWithTransform(features=features, labels=labels)
custom_dataset_with_transform_example[0]

(feature_1    2
 feature_2    3
 feature_3    6
 Name: 0, dtype: int64,
 label    1
 Name: 0, dtype: int64)

# Data Loader

It allows to retrieve **minibatches** from a `Dataset` object.

It can also reshuffle the data (`shuffle=True`) to avoid overfitting.

## Basic Usage

In [4]:
# Define batch size
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(fashion_mnist_train, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(fashion_mnist_test, batch_size=batch_size, shuffle=True)

for X, y in train_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


## DataLoader with CustomDataset

In [15]:
# Define batch size
batch_size = 2

# Create data loaders.
train_dataloader = DataLoader(custom_dataset_with_transform_example, batch_size=batch_size, shuffle=True)

for X, y in train_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

TypeError: default_collate: batch must contain tensors, numpy arrays, numbers, dicts or lists; found <class 'pandas.core.series.Series'>